# Silver: Flights
Parses Kafka Json payload, cleans, standardizes and deduplicates flight records
- **Source**: `bronze.flights` (raw json strings)
- **Target**: `silver.flights` (typed, cleaned, deduped)

## Import and Setup

In [0]:
email = dbutils.secrets.get(scope="flight-delay", key="db-email")

In [0]:

import sys
sys.path.insert(0, f"/Workspace/Users/{email}/ml-final-project/databricks")

from common.config import (
    BRONZE_FLIGHTS_PATH,
    abfss,
    SILVER_CONTAINER,
    configure_storage_auth,
)

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType,
)

SILVER_FLIGHTS_PATH = abfss(SILVER_CONTAINER, "flights")

## Authenticate with Azure Data Lake storage

In [0]:
configure_storage_auth(spark)

## Read Bronze

In [0]:

bronze = spark.read.format("delta").load(BRONZE_FLIGHTS_PATH)
print(f"Bronze rows: {bronze.count():,}")

## Schema Definition
Define the JSON schema for the `kafka_value` column.  
The ingestion wrapper in the producer adds these outer fields:
`ingestion_ts_utc, queried_airport, query_direction, chunk_from, chunk_to, source, payload`
The Aviation Edge payload itself is nested inside `payload`.

In [0]:

airport_schema = StructType([
    StructField("iataCode", StringType()),
    StructField("icaoCode", StringType()),
    StructField("terminal", StringType()),
    StructField("gate", StringType()),
    StructField("baggage", StringType()),
    StructField("delay", IntegerType()),
    StructField("scheduledTime", StringType()),
    StructField("estimatedTime", StringType()),
    StructField("actualTime", StringType()),
    StructField("estimatedRunway", StringType()),
    StructField("actualRunway", StringType()),
])

airline_schema = StructType([
    StructField("name", StringType()),
    StructField("iataCode", StringType()),
    StructField("icaoCode", StringType()),
])

flight_schema = StructType([
    StructField("number", StringType()),
    StructField("iataNumber", StringType()),
    StructField("icaoNumber", StringType()),
])

payload_schema = StructType([
    StructField("type", StringType()),
    StructField("status", StringType()),
    StructField("departure", airport_schema),
    StructField("arrival", airport_schema),
    StructField("airline", airline_schema),
    StructField("flight", flight_schema),
])

message_schema = StructType([
    StructField("ingestion_ts_utc", StringType()),
    StructField("queried_airport", StringType()),
    StructField("query_direction", StringType()),
    StructField("chunk_from", StringType()),
    StructField("chunk_to", StringType()),
    StructField("source", StringType()),
    StructField("payload", payload_schema),
])

## Parse JSON

In [0]:
parsed = (
    bronze
    .select(
        "kafka_key", "kafka_timestamp", "ingestion_ts_utc",
        F.from_json("kafka_value", message_schema).alias("msg")
    )
    .select(
        "kafka_key",
        "kafka_timestamp",
        F.col("ingestion_ts_utc").alias("bronze_ingestion_ts_utc"),
        "msg.*",
    )
)

## Transformations: Flatten, Clean and Standardize

In [0]:

TS_FMT = "yyyy-MM-dd'T'HH:mm:ss.SSS"  

def parse_flight_ts(col_name: str):
    """
    Parse the flight API timestamps robustly.
    Aviation Edge uses lowercase 't' sometimes
    """
    upper = F.upper(F.col(col_name))
    return F.to_timestamp(upper, TS_FMT)


silver = (
    parsed
    .select(
        # Identifiers
        F.upper(F.col("payload.flight.iataNumber")).alias("flight_iata_number"),
        F.upper(F.col("payload.flight.icaoNumber")).alias("flight_icao_number"),
        F.upper(F.col("payload.flight.number")).alias("flight_number"),

        # Airline
        F.upper(F.col("payload.airline.iataCode")).alias("airline_iata"),
        F.upper(F.col("payload.airline.icaoCode")).alias("airline_icao"),
        F.lower(F.col("payload.airline.name")).alias("airline_name"),

        # Departure airport
        F.upper(F.col("payload.departure.iataCode")).alias("dep_iata"),
        F.upper(F.col("payload.departure.icaoCode")).alias("dep_icao"),
        F.col("payload.departure.terminal").alias("dep_terminal"),
        F.col("payload.departure.gate").alias("dep_gate"),

        # Arrival airport
        F.upper(F.col("payload.arrival.iataCode")).alias("arr_iata"),
        F.upper(F.col("payload.arrival.icaoCode")).alias("arr_icao"),
        F.col("payload.arrival.terminal").alias("arr_terminal"),
        F.col("payload.arrival.gate").alias("arr_gate"),
        F.col("payload.arrival.baggage").alias("arr_baggage"),

        # Departure times (parsed)
        parse_flight_ts("payload.departure.scheduledTime").alias("dep_scheduled_ts"),
        parse_flight_ts("payload.departure.estimatedTime").alias("dep_estimated_ts"),
        parse_flight_ts("payload.departure.actualTime").alias("dep_actual_ts"),
        parse_flight_ts("payload.departure.estimatedRunway").alias("dep_estimated_runway_ts"),
        parse_flight_ts("payload.departure.actualRunway").alias("dep_actual_runway_ts"),

        # Arrival times (parsed)
        parse_flight_ts("payload.arrival.scheduledTime").alias("arr_scheduled_ts"),
        parse_flight_ts("payload.arrival.estimatedTime").alias("arr_estimated_ts"),
        parse_flight_ts("payload.arrival.actualTime").alias("arr_actual_ts"),
        parse_flight_ts("payload.arrival.estimatedRunway").alias("arr_estimated_runway_ts"),
        parse_flight_ts("payload.arrival.actualRunway").alias("arr_actual_runway_ts"),

        # Delays (may be null)
        F.col("payload.departure.delay").cast(IntegerType()).alias("dep_delay_minutes"),
        F.col("payload.arrival.delay").cast(IntegerType()).alias("arr_delay_minutes"),

        # Flight status 
        F.lower(F.col("payload.status")).alias("flight_status"),
        F.lower(F.col("payload.type")).alias("flight_type"),

        # Ingestion metadata 
        F.upper(F.col("queried_airport")).alias("queried_airport"),
        F.lower(F.col("query_direction")).alias("query_direction"),
        F.to_date("chunk_from").alias("chunk_from"),
        F.to_date("chunk_to").alias("chunk_to"),
        F.col("bronze_ingestion_ts_utc"),
        F.col("kafka_timestamp"),
    )
)

## Quality Filters
 Drop rows that are unusable for ML:
 - No flight identifier
 - No scheduled departure time
 - Test/invalid airports

In [0]:

silver_filtered = (
    silver
    .filter(F.col("flight_iata_number").isNotNull())
    .filter(F.col("dep_scheduled_ts").isNotNull())
    .filter(F.col("dep_iata").isNotNull())
    .filter(F.col("arr_iata").isNotNull())
)

print(f"After quality filter: {silver_filtered.count():,} rows")

## Deduplicate

In [0]:
from pyspark.sql.window import Window

dedup_window = (
    Window
    .partitionBy("flight_iata_number", "dep_scheduled_ts")
    .orderBy(F.col("bronze_ingestion_ts_utc").desc())
)

silver_deduped = (
    silver_filtered
    .withColumn("_rn", F.row_number().over(dedup_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

print(f"After dedup: {silver_deduped.count():,} rows")

## Derive the Target Variable
`dep_delay_minutes` comes directly from the API when available. When it's missing but we have both scheduled and actual times, compute it ourselves.

In [0]:
silver_final = (
    silver_deduped
    .withColumn(
        "dep_delay_minutes_computed",
        F.when(
            F.col("dep_delay_minutes").isNotNull(),
            F.col("dep_delay_minutes")
        ).otherwise(
            F.when(
                F.col("dep_actual_ts").isNotNull() & F.col("dep_scheduled_ts").isNotNull(),
                (
                    F.unix_timestamp("dep_actual_ts") -
                    F.unix_timestamp("dep_scheduled_ts")
                ) / 60
            ).otherwise(F.lit(None))
        ).cast(IntegerType())
    )
    
    .withColumn("dep_scheduled_date", F.to_date("dep_scheduled_ts"))
    .withColumn("dep_scheduled_hour", F.hour("dep_scheduled_ts"))
    .withColumn("dep_scheduled_day_of_week", F.dayofweek("dep_scheduled_ts"))
    .withColumn("dep_scheduled_month", F.month("dep_scheduled_ts"))
    # Partition by date for efficient queries later
    .withColumn("dep_scheduled_date_partition", F.to_date("dep_scheduled_ts"))
    .withColumn("silver_processed_ts_utc", F.current_timestamp())
)

## Write to Silver

In [0]:
(
    silver_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("dep_scheduled_date_partition")
    .save(SILVER_FLIGHTS_PATH)
)

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS silver.flights
    USING DELTA
    LOCATION '{SILVER_FLIGHTS_PATH}'
""")

### Test Silver Table

In [0]:
row_count = silver_final.count()
dbutils.notebook.exit(f'{{"rows_in_silver": {row_count}}}')